# Expertise: End-to-End Workflow

This notebook demonstrates the full pipeline:

1. Load an **OO-LD JSON input** (`example.oold.json`)
2. **Convert** it to RDF (rdflib native JSON-LD 1.1 parser)

The expertise schema is already user-friendly — the input is a plain JSON
object with URI arrays for each expertise category.  There is no simplified
transformation step.

---

## Environment setup

Create a virtual environment, run the install cell below, then re-select the kernel if needed:

```bash
python3 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
jupyter lab
```

JupyterLab will use the active venv automatically. Run the install cell below to add the required packages.

In [ ]:
%pip install -q rdflib pyyaml

In [ ]:
import sys
import json
import pathlib
import yaml
from importlib.metadata import version
import rdflib

print("Python :", sys.executable)
print()
print("rdflib  ", version("rdflib"))

# Paths
HERE   = pathlib.Path().resolve()   # notebooks/
SCHEMA = HERE.parent                # expertise/

---
## Step 1 — Load the OO-LD input

Edit `example.oold.json` (or point to your own file) to describe the expertise
of a specific person.  Each field is an array of URIs referencing knowledge
graph entities in the DSMS.

In [ ]:
doc = json.loads((SCHEMA / "example.oold.json").read_text())

print(json.dumps(doc, indent=2))

---
## Step 2 — Convert to RDF

rdflib ≥ 7 has a native JSON-LD 1.1 parser that reads the context directly from
`schema.oold.yaml` — no preprocessing or copy-paste needed.

In [ ]:
context = yaml.safe_load((SCHEMA / "schema.oold.yaml").read_text())["@context"]

g = rdflib.Dataset()
g.parse(data=json.dumps({"@context": context, **doc}), format="json-ld")

print(f"Graph contains {len(g)} triples.\n")
print(g.serialize(format="turtle"))

In [ ]:
# Optional: save to file
out_ttl = HERE / "output_expertise.ttl"
out_ttl.write_text(g.serialize(format="turtle"))
print(f"Written to {out_ttl}")

---
## Recap

| Step | Tool | Input | Output |
|---|---|---|---|
| Load | — | `example.oold.json` | Python dict |
| Convert to RDF | `rdflib` (JSON-LD 1.1) + `schema.oold.yaml` context | OO-LD dict | RDF graph |

To describe a different expert, edit `example.oold.json` with the relevant
knowledge graph URIs and re-run all cells.  No code changes needed.

> **No SHACL validation step:** this schema does not currently ship a
> `shape.ttl`.  Structural correctness is enforced by the JSON Schema in
> `schema.oold.yaml`.